# 04 — Frozen evaluation and acceptance review
Audit saved Phase 3/4 test evidence and results. Tables in this notebook are explicitly **archived results**, not a new model-training run. Notebooks 05–08 independently recompute predictions from the saved models.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import pandas as pd
from ml.phase4_models import sha256, select_model
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))
results = read_json('ml/artifacts/phase4/model_results.json')
lock_path = 'ml/artifacts/phase4/validation_selection.json'
lock = read_json(lock_path)
phase3 = read_json('reports/evidence/phase3_dataset_validation.json')
phase3_tests = read_json('reports/evidence/phase3_tests.json')
phase4_tests = read_json('reports/evidence/phase4_native_tests_final.json')
verification = read_json('reports/evidence/phase4_model_verification.json')
print('Interpreter:', sys.executable)

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe


## Metric definitions and selection
For n aligned rows, MAE = mean(|y−ŷ|), RMSE = sqrt(mean((y−ŷ)²)), and R² = 1 − sum((y−ŷ)²)/sum((y−mean(y))²). Undefined constant-target R² is represented explicitly.
Evaluate fractional predictions clipped to [0,12], without integer rounding. Raw metrics and clipping counts remain available. The fixed decision rule uses validation MAE first; test metrics only characterize the already selected model. Linear Regression can have better RMSE while persistence wins MAE.

In [2]:
assert sha256(lock_path) == results['selection_lock_sha256']
assert lock['test_opened'] is False
assert results['test_used_for_selection'] is False
assert results['models_refitted_on_validation'] is False
assert select_model(lock['validation_scores']) == lock['selected_model'] == results['selected_model'] == 'persistence'
archived = pd.DataFrame([{'partition': split, 'model': name, 'rows': score['rows'], **score['clipped'], 'clipped_count': score['clipped_count']} for split in ('validation', 'test') for name, score in results[split].items()])
display(archived)
assert archived.groupby('partition').rows.nunique().eq(1).all()
print('PASS: saved selection uses validation only; archived comparison has equal row counts')

,partition,model,rows,mae,rmse,r2,clipped_count
0,validation,persistence,10577,0.339983,0.684103,0.950447,0
1,validation,linear_regression,10577,0.414362,0.656363,0.954385,1302
2,validation,random_forest,10577,0.436999,0.680876,0.950914,0
3,test,persistence,10578,0.307714,0.648092,0.945254,0
4,test,linear_regression,10578,0.373729,0.613500,0.950942,1590
5,test,random_forest,10578,0.394675,0.641002,0.946445,0


PASS: saved selection uses validation only; archived comparison has equal row counts


In [3]:
assert all(item['status'] == 'PASS' for item in phase3['checks'].values())
assert phase3_tests['passed'] and phase4_tests['passed'] and verification['passed']
assert all(verification['checks'].values())
assert verification['data_model_bytes'] < 50_000_000
acceptance = pd.DataFrame([{'evidence': 'Phase 3 dataset checks', 'passed': len(phase3['checks']), 'failed': 0}, {'evidence': 'Phase 3 software tests (archived)', 'passed': phase3_tests['tests_run'], 'failed': phase3_tests['failures'] + phase3_tests['errors']}, {'evidence': 'Phase 4 software tests (archived)', 'passed': phase4_tests['tests_run'], 'failed': phase4_tests['failures'] + phase4_tests['errors']}, {'evidence': 'Phase 4 saved-model checks (archived)', 'passed': len(verification['checks']), 'failed': 0}])
display(acceptance)
display(pd.DataFrame(results['partitions']).T)
assert results['test_metrics_computed_twice_due_to_reload_recovery'] is True
assert results['retrained_during_finalization'] is False
print('PASS: archived evidence is internally consistent; no new training is claimed')

,evidence,passed,failed
0,Phase 3 dataset checks,18,0
1,Phase 3 software tests (archived),21,0
2,Phase 4 software tests (archived),21,0
3,Phase 4 saved-model checks (archived),28,0


,rows,first_origin,last_origin,last_target,origin_sha256
train,49360,2026-09-20 18:30:00+00:00,2026-10-25 01:09:00+00:00,2026-10-25 01:24:00+00:00,b837944ffdac8001f645a2c67d635ae70151cd07f8eeaf...
validation,10577,2026-10-25 01:25:00+00:00,2026-11-01 09:41:00+00:00,2026-11-01 09:56:00+00:00,6c28dca1155df5f02c545ae508f03e006079183464b664...
test,10578,2026-11-01 09:57:00+00:00,2026-11-08 18:14:00+00:00,2026-11-08 18:29:00+00:00,79f277c74ba6bd39467ba2bf0f72e0312f6795028c6d6a...


PASS: archived evidence is internally consistent; no new training is claimed


## Preserved failures, limitations and next steps
The original project environment had Application Control native-import denials. The approved isolated Windows ML runtime passed training. A strict forest reload equality check failed on floating-point summation order; verification used absolute 1e−12 tolerance with no retraining, retuning or new test-based selection. Earlier failed evidence is preserved.
Synthetic holdout performance cannot establish deployment accuracy, site demand, live sensor reliability or MQTT delivery. Phase 1 operator observations and Phase 2 transport tests retain their original limitations. No backend, firmware or dashboard integration is performed here.
Read the frozen Phase 3/4 PDFs and acceptance checklists under `reports/` for full methodology. Saved checks are reviewed here; the notebook execution evidence separately records which current notebook cells actually ran.